In [ ]:
#install thư viện pandas_profilling
!pip install ydata-profiling

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [50]:
!gdown 1KW6t_9VtCO4_s454u0bs2khpDK2_DU3y

Downloading...
From: https://drive.google.com/uc?id=1KW6t_9VtCO4_s454u0bs2khpDK2_DU3y
To: /content/dibetes.csv
100% 122k/122k [00:00<00:00, 10.8MB/s]


In [51]:
df = pd.read_csv("./content/dibetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,4.0,117.0,64.0,27.0,120.0,33.2,0.230,24.0,0.0
1,2.0,91.0,62.0,0.0,0.0,27.3,0.525,22.0,0.0
2,5.0,101.0,68.0,47.0,71.0,30.2,0.364,24.0,0.0
3,2.0,99.0,52.0,15.0,94.0,24.6,0.637,21.0,0.0
4,2.0,130.0,74.0,55.0,100.0,33.6,0.404,23.0,0.0



* Pregnancies: Number of times pregnant
* Glucose: Plasma glucose concentration a 2 hours in an oral glucose tolerance test
* BloodPressure: Diastolic blood pressure (mm Hg)
* SkinThickness: Triceps skin fold thickness (mm)
* Insulin: 2-Hour serum insulin (mu U/ml)
* BMI: Body mass index (weight in kg/(height in m)^2)
* DiabetesPedigreeFunction: Diabetes pedigree function
* Age: Age (years)
* Outcome: Class variable (0 or 1)


In [52]:
from ydata_profiling import ProfileReport
ProfileReport(df, title="Profiling Report")

Output hidden; open in https://colab.research.google.com to view.

In [53]:
df.isna().sum()

,0
Pregnancies,0
Glucose,5
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,11
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [54]:
from sklearn.model_selection import train_test_split
df = df.drop_duplicates(ignore_index=True)

X = df.drop(columns=["Outcome"], axis=1)
y = df.Outcome
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=10, train_size=0.8)

In [57]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler


zero_cols = ["BloodPressure", "Insulin", "SkinThickness"]
nan_cols = ["Glucose", "BMI"]

ct = ColumnTransformer(
    [
        ("nan_imputer", SimpleImputer(missing_values=np.nan, strategy="mean"), nan_cols),
        ("zero_imputer", SimpleImputer(missing_values=0, strategy="mean"), zero_cols)
    ],
    remainder="passthrough"
)

pipe_line = make_pipeline(ct, StandardScaler(), LogisticRegression())
pipe_line.fit(X_train, y_train)

print(classification_report(y_test, pipe_line.predict(X_test), digits=4))


              precision    recall  f1-score   support

         0.0     0.9328    0.9291    0.9310       254
         1.0     0.9032    0.9081    0.9057       185

    accuracy                         0.9203       439
   macro avg     0.9180    0.9186    0.9183       439
weighted avg     0.9203    0.9203    0.9203       439



### Finetuning Model By  Grid Search On Various Hyperparameters.

Dưới đây là danh sách các **siêu tham số (hyperparameters)** phổ biến cần được tinh chỉnh để đạt được độ khớp tốt nhất với dữ liệu. Chúng ta sẽ thử nhiều cấu hình siêu tham số khác nhau trên các cách chia **train/test** khác nhau để tìm ra mô hình phù hợp nhất — tức là mô hình có độ chính xác trên tập huấn luyện và tập kiểm tra **xấp xỉ nhau**, hoặc có **chênh lệch rất nhỏ** giữa hai độ chính xác này.

---

### 1. `hidden_layer_sizes`
- Nhận một **tuple các số nguyên**, biểu thị số lượng neuron trong các tầng ẩn của mạng **Multi-layer Perceptron (MLP)**.
- Số phần tử trong tuple tương ứng với số tầng ẩn; mỗi giá trị cho biết số perceptron trong tầng ẩn tương ứng.  
- `default = (100,)`

---

### 2. `activation`
- Xác định **hàm kích hoạt** cho các tầng ẩn (mặc định: `relu`).
- Các giá trị có thể:
  - `identity` – Không dùng hàm kích hoạt, \( f(x) = x \)
  - `logistic` – Hàm sigmoid logistic,  
    \( f(x) = \frac{1}{1 + \exp(-x)} \)
  - `tanh` – Hàm tang hyperbolic,  
    \( f(x) = \tanh(x) \)
  - `relu` – Hàm ReLU (Rectified Linear Unit),  
    \( f(x) = \max(0, x) \)

---

### 3. `solver`
- Xác định **thuật toán tối ưu** dùng để cập nhật trọng số của mạng nơ-ron.  
- `default = 'adam'`
- Các giá trị có thể:
  - `lbfgs`
  - `sgd`
  - `adam`

---



### 4. `learning_rate`
- Xác định **tốc độ học**.
- Chỉ áp dụng khi `solver = 'sgd'`.
- Các giá trị có thể:
  - `constant` – Giữ nguyên tốc độ học bằng `learning_rate_init`.
  - `invscaling` – Giảm dần tốc độ học theo thời gian:
  - `adaptive` – Giữ tốc độ học không đổi khi loss giảm hoặc score tăng. Nếu nhiều epoch liên tiếp không cải thiện (theo `tol`) sẽ dừng theo `early_stopping`

---

### 5. `batch_size`
- Xác định **kích thước mini-batch** dùng trong huấn luyện.
- Nhận giá trị số nguyên.  
- `default = 'auto'`
- Với chế độ `auto`:
  $\min(200, n_{\text{samples}})$

---

### 6. `tol`
- Ngưỡng hội tụ cho quá trình tối ưu.
- Nếu loss hoặc score không cải thiện ít nhất `tol` trong `n_iter_no_change` vòng lặp liên tiếp:
  - Dừng huấn luyện nếu `learning_rate = 'constant'`
  - Giảm tốc độ học nếu `learning_rate = 'adaptive'`
- `default = 0.0001`

---

### 7. `alpha`
- Hệ số phạt **L2 regularization** áp dụng lên trọng số.
- `default = 0.0001`

---

### 8. `momentum`
- Hệ số **momentum** cho gradient descent.
- Nhận giá trị trong khoảng \([0, 1]\).
- Chỉ áp dụng khi `solver = 'sgd'`.

---

### 9. `early_stopping`
- Xác định có **dừng sớm** quá trình huấn luyện hay không khi loss/score không còn cải thiện.
- Nhận giá trị boolean.  
- `default = False`

---

### 10. `validation_fraction`
- Tỷ lệ dữ liệu huấn luyện được tách ra làm **tập validation** khi `early_stopping` được bật.
- Nhận giá trị trong khoảng \([0, 1]\).  
- `default = 0.1`


In [58]:
from sklearn.neural_network import MLPClassifier

zero_cols = ["BloodPressure", "Insulin", "SkinThickness"]
nan_cols = ["Glucose", "BMI"]

ct = ColumnTransformer(
    [
        ("nan_imputer", SimpleImputer(missing_values=np.nan, strategy="mean"), nan_cols),
        ("zero_imputer", SimpleImputer(missing_values=0, strategy="mean"), zero_cols)
    ],
    remainder="passthrough"
)

pipe_line = make_pipeline(ct, StandardScaler(), MLPClassifier(hidden_layer_sizes=(100, ), activation="logistic", learning_rate="adaptive", batch_size=32, max_iter=200, random_state=42))

pipe_line.fit(X_train, y_train)

print(classification_report(y_test, pipe_line.predict(X_test), digits=4))


              precision    recall  f1-score   support

         0.0     0.9451    0.9488    0.9470       254
         1.0     0.9293    0.9243    0.9268       185

    accuracy                         0.9385       439
   macro avg     0.9372    0.9366    0.9369       439
weighted avg     0.9385    0.9385    0.9385       439



/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [69]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import numpy as np

ct = ColumnTransformer(
    [
        ("nan_imputer", SimpleImputer(missing_values=np.nan, strategy="mean"), nan_cols),
        ("zero_imputer", SimpleImputer(missing_values=0, strategy="mean"), zero_cols)
    ],
    remainder="passthrough"
)

# Pipeline
pipe = make_pipeline(
    ct,
    StandardScaler(),
    MLPClassifier(random_state=42, max_iter=300)  # tăng max_iter một chút
)

pipe.named_steps

{'columntransformer': ColumnTransformer(remainder='passthrough',
                   transformers=[('nan_imputer', SimpleImputer(),
                                  ['Glucose', 'BMI']),
                                 ('zero_imputer',
                                  SimpleImputer(missing_values=0),
                                  ['BloodPressure', 'Insulin',
                                   'SkinThickness'])]),
 'standardscaler': StandardScaler(),
 'mlpclassifier': MLPClassifier(max_iter=300, random_state=42)}

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import numpy as np

ct = ColumnTransformer(
    [
        ("nan_imputer", SimpleImputer(missing_values=np.nan, strategy="mean"), nan_cols),
        ("zero_imputer", SimpleImputer(missing_values=0, strategy="mean"), zero_cols)
    ],
    remainder="passthrough"
)

# Pipeline
pipe = make_pipeline(
    ct,
    StandardScaler(),
    MLPClassifier(random_state=42, max_iter=300)  # tăng max_iter một chút
)

# ==================== HYPERPARAMETER GRID ====================
param_dist = {
    'mlpclassifier__hidden_layer_sizes': [
        (50,), (100,), (100, 50), (100, 100), (50, 50, 50)
    ],
    'mlpclassifier__activation': ['relu', 'tanh', 'logistic'],
    'mlpclassifier__solver': ['adam', 'sgd'],
    'mlpclassifier__alpha': [0.0001, 0.001, 0.01, 0.05],
    'mlpclassifier__learning_rate': ['constant', 'adaptive'],
    'mlpclassifier__learning_rate_init': [0.001, 0.01, 0.005],
    'mlpclassifier__batch_size': [16, 32, 64],
    'mlpclassifier__early_stopping': [True, False]
}

# RandomizedSearchCV (nhanh hơn GridSearch)
search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring='f1',        # hoặc 'accuracy', 'roc_auc' tùy bài toán
    n_jobs=-1,
    random_state=42,
    verbose=2
)

# Train
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV score:", search.best_score_)

# Đánh giá trên test set
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred, digits=4))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params: {'mlpclassifier__solver': 'adam', 'mlpclassifier__learning_rate_init': 0.005, 'mlpclassifier__learning_rate': 'adaptive', 'mlpclassifier__hidden_layer_sizes': (100, 50), 'mlpclassifier__early_stopping': False, 'mlpclassifier__batch_size': 32, 'mlpclassifier__alpha': 0.05, 'mlpclassifier__activation': 'tanh'}
Best CV score: 0.8988537467864385
              precision    recall  f1-score   support

         0.0     0.9447    0.9409    0.9428       254
         1.0     0.9194    0.9243    0.9218       185

    accuracy                         0.9339       439
   macro avg     0.9320    0.9326    0.9323       439
weighted avg     0.9340    0.9339    0.9340       439



#California Housing Dataset – Mô tả chi tiết

## 1. Tổng quan
**California Housing Dataset** là một bộ dữ liệu chuẩn trong *scikit-learn*, được sử dụng rộng rãi cho các bài toán **học máy có giám sát – hồi quy**.  
Mục tiêu của dataset là **dự đoán giá nhà trung vị** tại các khu dân cư ở bang California (Hoa Kỳ).

Dataset này thường được dùng trong:
- Giảng dạy Machine Learning
- Thực hành regression
- So sánh các mô hình học máy
- Minh họa preprocessing và evaluation

---

## 2. Nguồn gốc dữ liệu
- Dữ liệu được trích xuất từ **1990 U.S. Census**
- Thu thập thông tin về **các census block group** tại California
- Mỗi mẫu đại diện cho **một block dân cư**, không phải một căn nhà riêng lẻ

---

## 3. Quy mô dataset

| Thuộc tính | Giá trị |
|-----------|--------|
| Số lượng mẫu (samples) | **20,640** |
| Số lượng đặc trưng (features) | **8** |
| Kiểu dữ liệu | Numeric (số thực) |
| Giá trị thiếu (missing values) |  Không có |
| Loại bài toán | Supervised Learning – Regression |

---

## 4. Các feature

| Feature | Mô tả chi tiết |
|-------|---------------|
| **MedInc** | Thu nhập trung vị của các hộ gia đình trong block, đơn vị là **10,000 USD** |
| **HouseAge** | Tuổi trung bình của các ngôi nhà trong block (tính bằng năm) |
| **AveRooms** | Số phòng trung bình trên mỗi hộ gia đình |
| **AveBedrms** | Số phòng ngủ trung bình trên mỗi hộ gia đình |
| **Population** | Tổng số dân cư sinh sống trong block |
| **AveOccup** | Số người trung bình trên mỗi hộ gia đình |
| **Latitude** | Vĩ độ địa lý của block |
| **Longitude** | Kinh độ địa lý của block |

**Lưu ý**:
- Tất cả các feature đều là **numeric**
- `Latitude` và `Longitude` mang thông tin **không gian địa lý**, rất quan trọng trong dự đoán giá nhà

---

## 5. Target

| Target | Mô tả |
|------|------|
| **MedHouseVal** | Giá nhà trung vị của block, đơn vị là **100,000 USD** |

### Ví dụ:
- `MedHouseVal = 2.0` → Giá nhà trung vị ≈ **200,000 USD**
- `MedHouseVal = 5.0` → Giá nhà trung vị ≈ **500,000 USD**

Target này là **biến liên tục**, do đó bài toán là **hồi quy (regression)**.

---

## 6. Cách tải dataset bằng scikit-learn

```python
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
X = data.data        # Feature matrix
y = data.target      # Target

#Lấy tên các feature
feature_names = data.feature_names
print(feature_names)

['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
 'Population', 'AveOccup', 'Latitude', 'Longitude']



In [ ]:
from sklearn.datasets import fetch_california_housing
california = fetch_california_housing()
x_california = california.data
y_california = california.target
print("Dataset Sizes ",x_california.shape, y_california.shape)

Dataset Sizes  (20640, 8) (20640,)


In [ ]:
california

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]]),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894]),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': '.. _california_housing_dataset:\n

In [ ]:
# Spliting dataset into train and test dataset
x_train, x_test, y_train, y_test = train_test_split(x_california, y_california, test_size = 0.25, random_state = 42)

# MLPRegressor
`MLPRegressor`is an estimator available as a part of the `neural_network` module of sklearn for performing regression tasks using a multi-layer perceptron.

In [ ]:
# import the regressor
from sklearn.neural_network import MLPRegressor
reg = MLPRegressor(activation = 'relu', hidden_layer_sizes = (150, 50, 100), learning_rate= 'constant', solver= 'adam',random_state = 42)
reg.fit(x_train,y_train)

MLPRegressor(hidden_layer_sizes=(150, 50, 100), random_state=42)

In [ ]:
y_preds = reg.predict(x_test)

print(y_preds[:5])
print(y_test[:5])

print("Train Score",reg.score(x_train,y_train))
print("Test Score" , reg.score(x_test,y_test))


[0.65726675 1.35398462 2.21797907 2.285909   1.93995645]
[0.477   0.458   5.00001 2.186   2.78   ]
Train Score 0.4547213914890941
Test Score 0.4524277137662601


In [ ]:
print("Number of Coefficents :", len(reg.coefs_))
[weights.shape for weights in reg.coefs_]


Number of Coefficents : 4


[(8, 150), (150, 50), (50, 100), (100, 1)]

In [ ]:
[weights for weights in reg.coefs_]

In [ ]:
print("Number of intecepts :",len(reg.intercepts_))
[intercepts.shape for intercepts in reg.intercepts_]

Number of intecepts : 4


[(150,), (50,), (100,), (1,)]

In [ ]:
print("Number of iterations estimators run: ", reg.n_iter_)
print("name of output layer activation function: ", reg.out_activation_)

Number of iterations estimators run:  33
name of output layer activation function:  identity


### Finetuning Model By Doing Grid Search




In [ ]:
%%time
reg= MLPRegressor(random_state = 42)
params= {'activation': ['relu','identity','tanh','logistic'],
        'hidden_layer_sizes': [50,100,150],
         'solver' : ['lbfgs','adam'],
         'learning_rate': ['constant','adaptive']
        }

reg_grid = GridSearchCV(reg,param_grid = params,n_jobs= -1,verbose = 10,cv=5)
reg_grid.fit(x_train,y_train)



Fitting 5 folds for each of 72 candidates, totalling 360 fits
CPU times: user 12.3 s, sys: 2.21 s, total: 14.5 s
Wall time: 34min 17s


GridSearchCV(cv=5, estimator=MLPRegressor(random_state=42), n_jobs=-1,
             param_grid={'activation': ['relu', 'identity', 'tanh', 'logistic'],
                         'hidden_layer_sizes': [50, 100, 150],
                         'learning_rate': ['constant', 'adaptive',
                                           'invscaling'],
                         'solver': ['lbfgs', 'adam']},
             verbose=10)

In [ ]:
print("Train score: ", reg_grid.score(x_train,y_train))
print("Test score: ", reg_grid.score(x_test,y_test))
print("Best R2 Score by grid search: ",reg_grid.best_score_)
print("Best Parameters: ", reg_grid.best_params_)
print("Best Estimators: ",reg_grid.best_estimator_)

Train score:  0.6380751395918024
Test score:  0.628058186074824
Best R2 Score by grid search:  0.6538801802790248
Best Parameters:  {'activation': 'logistic', 'hidden_layer_sizes': 50, 'learning_rate': 'constant', 'solver': 'adam'}
Best Estimators:  MLPRegressor(activation='logistic', hidden_layer_sizes=50, random_state=42)
